In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

import nltk
from nltk.corpus import stopwords

In [2]:
df = pd.read_csv("reviews.csv")
df["review_text"].head(20)

0                                             It's good
1     WhatsApp not working well always shows offline...
2     Oppo not corresponding, share with me the offi...
3     Excellent app, great communication super conne...
4          simply the ɓest for chat and calls.i love it
5            good. but i need WhatsApp premium features
6     learning learning learning learning learning l...
7       Awesome. I just need it to download and install
8                            very nice app thnx so much
9                          Really really apriacite 100/
10    whatsapp web not working camera Focus not working
11                                 Nice and good to use
12    This account can no longer use watsapp due to ...
13    forever controversy workshop lawyer carbon ins...
14    I've been using this app for years and I just ...
15    AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA...
16    They banned my account more than 9 times I did...
17    i can't open my whatsapp whenever i open i

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6210 entries, 0 to 6209
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   review_id    6210 non-null   str  
 1   rating       6210 non-null   int64
 2   review_text  6210 non-null   str  
 3   review_date  6210 non-null   str  
 4   helpful      6210 non-null   int64
dtypes: int64(2), str(3)
memory usage: 242.7 KB


In [4]:
#Text preprocessing
stop_words = set(stopwords.words("english"))

words_to_keep = {
    "not", "no", "nor", "never", "neither", "nothing", "very", "too", "most", "least",
    "more", "less", "least", "few", "but", "however", "although", "just", "only", "above", "below"
}

stop_words = stop_words - words_to_keep

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"\S+@\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()

    words= ([word for word in text.split() if word not in stop_words])
    
    return " ".join(words)


df["clean_text"] = df["review_text"].apply(clean_text)

df = df[df["clean_text"].str.len() > 0].reset_index(drop = True)

print("\nRemaning rows:", df.shape)


Remaning rows: (6168, 6)


In [5]:
df["clean_text"].head(20)

0                                                  good
1     whatsapp not working well always shows offline...
2       oppo not corresponding share official app oppor
3     excellent app great communication super connec...
4                           simply est chat callsi love
5               good but need whatsapp premium features
6     learning learning learning learning learning l...
7                    awesome just need download install
8                               very nice app thnx much
9                               really really apriacite
10    whatsapp web not working camera focus not working
11                                        nice good use
12    account no longer use watsapp due spam kyu ho ...
13    forever controversy workshop lawyer carbon ins...
14    ive using app years just absolutely love featu...
15    aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa...
16                 banned account more times didnt know
17    cant open whatsapp whenever open close aut

In [6]:
#Convert text to numeric
vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1,3),
    min_df=3,
    max_df=0.9,
    sublinear_tf=True
)

In [7]:
#Split the data for training
x_raw = df["clean_text"]
y = df["rating"]

x_train_raw, x_test_raw, y_train, y_test = train_test_split(
    x_raw, y,
    test_size = 0.2,
    random_state = 42
)

# Fit vectorizer ONLY on training data to avoid data leakage
x_train = vectorizer.fit_transform(x_train_raw)
x_test = vectorizer.transform(x_test_raw)


In [8]:
#Setting the parameters for the model and training it
import lightgbm as lgb

model = lgb.LGBMClassifier(
    n_estimators = 300,
    learning_rate = 0.01,
    num_leaves = 31,
    max_depth = -1,
    verbose = -1  # Suppress verbose output
)

model.fit(x_train, y_train)


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.01
,n_estimators,300
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [9]:
y_prediction = model.predict(x_test)

print(f"Test Accuracy: {accuracy_score(y_test, y_prediction):.4f}\n")

target_names = [f"Rating {i}" for i in range(1, 6)]
print(classification_report(y_test, y_prediction, target_names=target_names, zero_division=0))


Test Accuracy: 0.5932

              precision    recall  f1-score   support

    Rating 1       0.58      0.60      0.59       319
    Rating 2       0.18      0.02      0.04        92
    Rating 3       0.00      0.00      0.00       110
    Rating 4       0.14      0.03      0.04       117
    Rating 5       0.62      0.90      0.74       596

    accuracy                           0.59      1234
   macro avg       0.31      0.31      0.28      1234
weighted avg       0.48      0.59      0.51      1234



/home/gromit/INFO284/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [10]:
feature_names = vectorizer.get_feature_names_out()

importance = pd.DataFrame({
    "feature": feature_names,
    "importance": model.feature_importances_
}).sort_values(by = "importance", ascending = False)

print(importance.head(20))

       feature  importance
3959  whatsapp        1760
178        app        1168
2367       not         920
3806      very         905
1530      good         890
2662    please         822
2072      love         740
2324      nice         694
613       cant         668
544        but         652
437       best         598
1585     great         587
23     account         560
3738       use         527
1996      like         493
1091  download         407
3566      time         379
2338        no         366
689       chat         362
2883    really         360


In [11]:
#Grid search to find the best parameters for the model
from sklearn.model_selection import GridSearchCV

# Use a smaller, more focused grid to keep computation feasible
model_gs = lgb.LGBMClassifier(verbose=-1)

param_grid = {
    "n_estimators": [100, 300],
    "learning_rate": [0.01, 0.1],
    "num_leaves": [31, 63],
}

grid = GridSearchCV(
    estimator = model_gs,
    param_grid = param_grid,
    cv = 3,
    scoring = "accuracy",
    verbose = 1,
    n_jobs = -1
)

grid.fit(x_train, y_train)
print("Best parameters:", grid.best_params_)
print("Best CV accuracy:", grid.best_score_)


Fitting 3 folds for each of 8 candidates, totalling 24 fits


/home/gromit/INFO284/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/gromit/INFO284/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/gromit/INFO284/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/gromit/INFO284/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/gromit/INFO284/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with featur

Best parameters: {'learning_rate': 0.01, 'n_estimators': 100, 'num_leaves': 63}
Best CV accuracy: 0.5772206568603525
